In [1]:
#to prevent colab to automatically disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Step 1: Importing Libraries**

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd
import numpy as np

**Step 2: Loading Dataset**

In [5]:
columns_needed2 = ['DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME', 'Crm Cd',
                    'Crm Cd Desc']
columns_needed1 = ['DATE OCC', 'TIME OCC', 'AREA ', 'AREA NAME', 'Crm Cd',
                    'Crm Cd Desc']

In [6]:
dataset_path1 = r'/content/drive/MyDrive/Crime Prediction/LA/Original Dataset/Crime Dataset/Crime_Data_from_2010_to_2019.csv'
dataset_path2 = r'/content/drive/MyDrive/Crime Prediction/LA/Original Dataset/Crime Dataset/Crime_Data_from_2020_to_Present.csv'

dataset1= pd.read_csv(dataset_path1,usecols = columns_needed1)
dataset2= pd.read_csv(dataset_path2,usecols = columns_needed2)

In [7]:
dataset1.shape

(2153259, 6)

In [8]:
dataset2.shape

(1005149, 6)

**Step 3: Merging Dataset**

In [9]:
dataset1.columns = dataset1.columns.str.strip()
dataset2.columns = dataset2.columns.str.strip()

dataset = pd.concat([dataset1[dataset1.columns], dataset2[dataset2.columns]], ignore_index=True)
print(f"Final Merged Dataset Shape: {dataset.shape}")
print("Columns in Merged Dataset:")
print(dataset.columns)

Final Merged Dataset Shape: (3158408, 6)
Columns in Merged Dataset:
Index(['DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME', 'Crm Cd', 'Crm Cd Desc'], dtype='object')


In [10]:
dataset.shape

(3158408, 6)

**Step 4: Data Formatting**

In [11]:
# Convert 'DATE OCC' to proper datetime format
dataset["DATE OCC"] = pd.to_datetime(dataset["DATE OCC"], format="mixed",errors = 'coerce')

# Ensure 'TIME OCC' is a string with 4 digits (e.g., '915' → '0915')
dataset["TIME OCC"] = dataset["TIME OCC"].astype(str).str.zfill(4)

# Extract hours and minutes
dataset["HOUR"] = dataset["TIME OCC"].str[:2].astype(int)  # First 2 digits = Hour
dataset["MINUTE"] = dataset["TIME OCC"].str[2:].astype(int)  # Last 2 digits = Minutes

# Create final 'DATETIME' column
dataset["DATETIME"] = dataset["DATE OCC"] + pd.to_timedelta(dataset["HOUR"], unit="h") + pd.to_timedelta(dataset["MINUTE"], unit="m")

# Drop unnecessary columns
dataset.drop(columns=["DATE OCC","TIME OCC", "HOUR", "MINUTE"], inplace=True)

# Display the cleaned dataset
print(dataset[["DATETIME"]].head())


             DATETIME
0 2010-02-20 13:50:00
1 2010-09-12 00:45:00
2 2010-08-09 15:15:00
3 2010-01-05 01:50:00
4 2010-01-02 21:00:00


**Step 5: Mapping Crime Descriptions**

In [12]:
# Define crime categories based on words from 142 descriptions
crime_mapping = {
    "HOMICIDE": ["homicide", "murder", "manslaughter"],
    "SEX OFFENSE": ["rape", "sexual", "indecent", "child pornography", "lewd", "incest", "sodomy", "molestation"],
    "ASSAULT": ["assault", "battery", "stalking", "threat", "intimidation", "domestic violence"],
    "KIDNAPPING": ["kidnap", "child stealing", "abduction"],
    "WEAPONS VIOLATION": ["weapon", "firearm", "gun", "bomb", "shots fired"],
    "ROBBERY": ["robbery", "armed robbery", "attempted robbery", "purse snatching", "pickpocket"],
    "BURGLARY": ["burglary", "break-in", "home invasion"],
    "LARCENY/THEFT": ["theft", "shoplifting", "embezzlement", "fraud", "vandalism", "arson", "drunk roll", "coin machine"],
    "CRIMINAL TRESPASS": ["trespassing", "prowler"],
    "DRUG/NARCOTIC": ["drug", "narcotic", "substance", "possession", "illegal substance"],
    "GAMBLING": ["gambling", "betting"],
    "PROSTITUTION": ["prostitution", "pandering", "pimping"],
    "PUBLIC PEACE VIOLATION": ["riot", "disturbing", "disorderly", "peace", "resisting"],
    "LIQUOR LAW VIOLATION": ["liquor", "drunk", "intoxicated"],
    "PUBLIC INDECENCY": ["public indecency", "obscenity", "flashing", "indecent exposure"],
    "ARSON": ["arson", "fire", "burning"],
    "HUMAN TRAFFICKING": ["human trafficking", "forced labor"],
    "DECEPTIVE PRACTICE": ["fraud", "scam", "identity theft", "counterfeit"],
    "OBSCENITY": ["obscenity", "lewd conduct"],
    "RITUALISM": ["ritual", "cult", "sacrifice"],
    "INTERFERENCE WITH PUBLIC OFFICER": ["resisting arrest", "contempt of court"],
    "NON-CRIMINAL": ["firearm restraining order", "emergency protective order"],
    "OTHER OFFENSES": ["miscellaneous crime", "violation of restraining order", "threatening phone calls"],
}

def categorize_crime(crime_description):
    """Categorizes a crime description into the most relevant Chicago crime type."""
    crime_description = crime_description.lower()  # Convert to lowercase

    for crime_type, keywords in crime_mapping.items():
        if any(word in crime_description for word in keywords):
            return crime_type  # Assign first matching category

    return "OTHER OFFENSES"  # Default if no match is found


# Apply categorization
dataset["Crime Type"] = dataset["Crm Cd Desc"].apply(categorize_crime)


In [13]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3158408 entries, 0 to 3158407
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   AREA         int64         
 1   AREA NAME    object        
 2   Crm Cd       int64         
 3   Crm Cd Desc  object        
 4   DATETIME     datetime64[ns]
 5   Crime Type   object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 144.6+ MB


In [14]:
dataset.describe()

,AREA,Crm Cd,DATETIME
count,3.158408e+06,3.158408e+06,3158408
mean,1.101049e+01,5.049524e+02,2017-06-27 14:01:19.648619520
min,1.000000e+00,1.100000e+02,2010-01-01 00:01:00
25%,6.000000e+00,3.300000e+02,2013-12-01 00:01:00
50%,1.100000e+01,4.420000e+02,2017-06-19 10:23:30
75%,1.600000e+01,6.260000e+02,2021-01-30 09:00:00
max,2.100000e+01,9.560000e+02,2025-03-13 17:57:00
std,6.044820e+00,2.090118e+02,NaN


In [15]:
dataset.shape

(3158408, 6)

In [16]:
dataset.isnull().sum()

,0
AREA,0
AREA NAME,0
Crm Cd,0
Crm Cd Desc,0
DATETIME,0
Crime Type,0


**Step 6: Taking Necessary Feature**

In [17]:
#Reordering Dataset
dataset = dataset[['DATETIME', 'AREA', 'AREA NAME','Crime Type']]

In [18]:
dataset.head()

,DATETIME,AREA,AREA NAME,Crime Type
0,2010-02-20 13:50:00,13,Newton,OTHER OFFENSES
1,2010-09-12 00:45:00,14,Pacific,LARCENY/THEFT
2,2010-08-09 15:15:00,13,Newton,OTHER OFFENSES
3,2010-01-05 01:50:00,6,Hollywood,OTHER OFFENSES
4,2010-01-02 21:00:00,1,Central,SEX OFFENSE


In [19]:
dataset.tail()

,DATETIME,AREA,AREA NAME,Crime Type
3158403,2025-01-17 15:30:00,21,Topanga,WEAPONS VIOLATION
3158404,2025-02-21 15:30:00,3,Southwest,OTHER OFFENSES
3158405,2025-02-13 21:00:00,3,Southwest,OTHER OFFENSES
3158406,2025-01-14 12:50:00,5,Harbor,ROBBERY
3158407,2025-02-27 15:50:00,16,Foothill,OTHER OFFENSES


In [21]:
saving_path =  r'/content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/1.Merging/Crime_Dataset.csv'
dataset.to_csv(saving_path, index=False)